# Climate Discourse Data Curation and Classification Pipeline (1996–2024)

This notebook documents the complete data construction workflow for building a structured panel dataset of climate-related discourse from central bank speeches. The objective is to extract, validate, and classify climate-relevant sentences in order to construct a reproducible measure of climate communication intensity at the country–year level.

The workflow proceeds in the following stages:

1. **Data Acquisition**
   - Central bank speeches are downloaded programmatically from the BIS database using the `gingado` dataset interface.
   - Raw speech metadata include URL, title, description, author, date, and full text.

2. **Pre-processing and Temporal Structuring**
   - Speech dates are parsed into standardized datetime format.
   - Year identifiers are constructed for panel aggregation.
   - Initial descriptive statistics (speeches per year) are generated to verify temporal coverage.

3. **Lexicon-Based Climate Sentence Extraction**
   - A curated climate-finance lexicon is imported.
   - Sentence segmentation is performed using spaCy’s sentencizer.
   - Sentences are matched against lexicon terms using token-safe rules:
     - Single-word terms require exact boundary matches.
     - Multi-word terms require all component tokens to be present within the sentence.
   - Extracted sentences are saved with associated metadata and matched keyword.

4. **Deduplication**
   - Because multiple keywords may match the same sentence, duplicate sentences are removed.
   - A unique sentence file is created to prevent artificial inflation of discourse counts.

5. **Transformer-Based Validation (ClimateBERT)**
   - Unique sentences are classified using the pretrained model `climatebert/distilroberta-base-climate-detector`.
   - For each sentence, the model produces:
     - Predicted label (climate / non-climate)
     - Posterior probability scores
   - Only sentences classified as climate-related are retained for final analysis.

6. **Final Cleaning and Validation**
   - Dates are re-validated using a two-pass parsing strategy.
   - The final climate-only sentence dataset is restricted to the 1996–2024 study window.
   - Total counts of unique climate-related sentences are verified.

The resulting output forms the foundation for constructing country–year measures of climate discourse intensity used in subsequent econometric analysis. The full pipeline is designed to ensure transparency, replicability, and methodological robustness through the combination of lexicon-based retrieval and context-aware transformer classification.

In [ ]:
# !pip install gingado

from gingado.datasets import load_CB_speeches

all_speeches = load_CB_speeches()
all_speeches.to_csv("central_bank_speeches.csv", index=False)

print("Dataset downloaded successfully.")

Dataset downloaded successfully.


In [ ]:
import pandas as pd
from pathlib import Path

input_path = Path("central_bank_speeches.csv")
df = pd.read_csv(input_path)

# parse ISO datetime correctly
df["date"] = pd.to_datetime(df["date"], errors="coerce")  # dayfirst not needed
df["year"] = df["date"].dt.year

# speeches per year
year_table = (
    df["year"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)
year_table.columns = ["Year", "Number_of_Speeches"]

print(year_table.to_string(index=False))
print("\nTotal parsed speeches:", int(year_table["Number_of_Speeches"].sum()))

 Year  Number_of_Speeches
 1996                  10
 1997                 212
 1998                 204
 1999                 281
 2000                 298
 2001                 313
 2002                 337
 2003                 338
 2004                 572
 2005                 599
 2006                 717
 2007                 836
 2008                 871
 2009                 848
 2010                 949
 2011                 944
 2012                 920
 2013                1003
 2014                 936
 2015                 866
 2016                 911
 2017                1022
 2018                 923
 2019                1011
 2020                 711
 2021                 672
 2022                 761
 2023                 742
 2024                 816
 2025                 741

Total parsed speeches: 20364


# Climate finance sentence extraction
Inputs
   - climate_finance_lexicon.csv  (column: term)
   - central_bank_speeches.csv    (columns include: url,title,description,date,text,author)
Output:
   - climate_finance_sentences.csv (columns: url,title,description,date,author,keyword,sentence)

Matching rules:
   - Sentence splitting via spaCy sentencizer
   - 1-word term: word must appear in the sentence (token boundary safe)
   - Multi-word term: ALL component words must appear in the sentence (any order)
   - One row per (keyword hit × sentence). Multiple keywords per sentence => multiple rows.
   - Exact duplicate rows removed at the end (same url + keyword + sentence).


In [ ]:
# Dependencies
# !pip install -U spacy

import re
import pandas as pd
import spacy
from collections import defaultdict

LEXICON_PATH = "climate_lexicon.csv"
SPEECHES_PATH = "central_bank_speeches.csv"
OUT_PATH = "climate_finance_sentences.csv"

# -------------------------
# Load data
# -------------------------
lex = pd.read_csv(LEXICON_PATH)
speeches = pd.read_csv(SPEECHES_PATH)

if "term" not in lex.columns:
    raise ValueError("Lexicon file must contain a column named: term")
if "text" not in speeches.columns:
    raise ValueError("Speeches file must contain a column named: text")

for col in ["url", "title", "description", "date", "author"]:
    if col not in speeches.columns:
        speeches[col] = ""

# -------------------------
# spaCy (fast)
# -------------------------
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")
nlp.max_length = max(nlp.max_length, 20_000_000)

_ws_re = re.compile(r"\s+")
def normalize_text(x: str) -> str:
    if x is None:
        return ""
    x = str(x).replace("\u00a0", " ")
    x = _ws_re.sub(" ", x).strip()
    return x

# -------------------------
# Term preprocessing
# -------------------------
def term_to_words(term: str):
    """
    Convert a term into normalized component words.
    Keeps only alphanumeric tokens, lowercased.
    Examples:
      "emissions trading" -> ["emissions", "trading"]
      "net-zero" -> ["net", "zero"]
      "TCFD" -> ["tcfd"]
    """
    term = normalize_text(term).lower()
    # keep word-like pieces; split hyphens/punct into separate words
    parts = re.findall(r"[a-z0-9]+", term)
    return parts

terms = (
    lex["term"].astype(str).map(lambda x: x.strip()).replace({"": None, "nan": None}).dropna().drop_duplicates().tolist()
)

term_words = {}
for t in terms:
    w = term_to_words(t)
    if len(w) == 0:
        continue
    term_words[t] = w

print(f"Lexicon terms loaded: {len(term_words):,}")
print(f"Speeches loaded: {len(speeches):,}")

# Build inverted index: word -> set(terms containing that word)
word_to_terms = defaultdict(set)
for t, words in term_words.items():
    for w in set(words):
        word_to_terms[w].add(t)

# -------------------------
# Extraction
# -------------------------
records = []
texts = speeches["text"].map(normalize_text).tolist()

def sent_token_set(sent_doc):
    """
    Token-boundary safe set of words in the sentence.
    Uses spaCy tokenization + lowercase + alnum filter.
    """
    s = set()
    for tok in sent_doc:
        if tok.is_space:
            continue
        # keep only alphanumeric tokens (drop pure punctuation)
        if tok.is_alpha or tok.is_digit or tok.like_num:
            s.add(tok.lower_)
        else:
            # for tokens like "net-zero" spaCy may keep "-", so also split
            for p in re.findall(r"[a-z0-9]+", tok.lower_):
                if p:
                    s.add(p)
    return s

for i, doc in enumerate(nlp.pipe(texts, batch_size=16)):
    if not doc.text:
        continue

    row = speeches.iloc[i]
    meta = {
        "url": row.get("url", ""),
        "title": row.get("title", ""),
        "description": row.get("description", ""),
        "date": row.get("date", ""),
        "author": row.get("author", ""),
    }

    for sent in doc.sents:
        sent_text = sent.text.strip()
        if not sent_text:
            continue

        tokenset = sent_token_set(sent)

        # Candidate terms: union of terms that share at least one word with this sentence
        candidate_terms = set()
        for w in tokenset:
            candidate_terms |= word_to_terms.get(w, set())

        if not candidate_terms:
            continue

        # Verify term rule: all words from the term must exist in the sentence
        for t in candidate_terms:
            words = term_words[t]
            if all(w in tokenset for w in words):
                records.append(
                    {
                        **meta,
                        "keyword": t,          # keep original term text
                        "sentence": sent_text,
                    }
                )

out = pd.DataFrame.from_records(
    records,
    columns=["url", "title", "description", "date", "author", "keyword", "sentence"],
)

print(f"Raw extracted rows (keyword × sentence): {len(out):,}")

out = out.drop_duplicates(subset=["url", "keyword", "sentence"]).reset_index(drop=True)

print(f"After de-duplication (url+keyword+sentence): {len(out):,}")

out.to_csv(OUT_PATH, index=False, encoding="utf-8")
print(f"Saved: {OUT_PATH}")

Lexicon terms loaded: 887
Speeches loaded: 20,364
Raw extracted rows (keyword × sentence): 80,540
After de-duplication (url+keyword+sentence): 80,454
Saved: climate_finance_sentences.csv


In [3]:
import pandas as pd

df = pd.read_csv("climate_finance_sentences.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

total_sentences = df[
    df["date"].dt.year.between(1996, 2024)
]["sentence"].dropna().shape[0]

print("Total sentences (1996–2024):", total_sentences)

Total sentences (1996–2024): 75583


In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# Keep UNIQUE sentences (avoid inflation from multiple keywords)
# - Input/Output are in the same folder as the notebook
# =========================================================

IN_PATH  = "climate_finance_sentences.csv"                 #  extracted file (keyword × sentence)
OUT_PATH = "climate_finance_sentences_unique.csv"          # output: unique sentences with metadata

# -------------------------
# Load
# -------------------------
df = pd.read_csv(IN_PATH)

required = ["sentence"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found columns: {list(df.columns)}")

# -------------------------
# Counts: total / unique / duplicates
# -------------------------
total_rows = len(df)

# Unique by sentence text only (this removes multiple-keyword hits)
unique_sentences = df["sentence"].astype(str).str.strip().nunique(dropna=True)

dup_rows = total_rows - unique_sentences

print("=== Sentence inflation check (keyword hits vs unique sentences) ===")
print(f"Total extracted rows (keyword × sentence): {total_rows:,}")
print(f"Unique sentences (by sentence text):      {unique_sentences:,}")
print(f"Duplicate rows removed if unique-only:     {dup_rows:,}")

# -------------------------
# Build unique-sentence dataset with ALL columns retained
# Strategy:
# - Sort to keep the "best" representative row per sentence.
# - Keep the first row per sentence after sorting.
#
#   1) keep row with non-empty date
#   2) then non-empty title
#   3) then non-empty url
# -------------------------
def nonempty(x):
    return x.notna() & (x.astype(str).str.strip() != "")

df_work = df.copy()
df_work["sentence_norm"] = df_work["sentence"].astype(str).str.strip()

# Add preference flags (so we keep more informative metadata if duplicates exist)
for col in ["date", "title", "url", "author", "description", "keyword"]:
    if col not in df_work.columns:
        df_work[col] = ""
    df_work[f"has_{col}"] = nonempty(df_work[col]).astype(int)

# Sort so the kept row per sentence tends to have richer metadata
sort_cols = [f"has_{c}" for c in ["date", "title", "url", "author", "description", "keyword"]]
df_work = df_work.sort_values(sort_cols, ascending=False)

# Drop duplicates by normalized sentence, keep first "best" row
df_unique = df_work.drop_duplicates(subset=["sentence_norm"], keep="first").drop(columns=["sentence_norm"] + sort_cols)

# -------------------------
# Save
# -------------------------
df_unique.to_csv(OUT_PATH, index=False, encoding="utf-8")

print("\n=== Saved unique sentence dataset ===")
print(f"Rows saved (unique sentences): {len(df_unique):,}")


=== Sentence inflation check (keyword hits vs unique sentences) ===
Total extracted rows (keyword × sentence): 80,454
Unique sentences (by sentence text):      50,251
Duplicate rows removed if unique-only:     30,203

=== Saved unique sentence dataset ===
Rows saved (unique sentences): 50,251


In [6]:
import pandas as pd

df = pd.read_csv("climate_finance_sentences_unique.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

total_sentences = df[
    df["date"].dt.year.between(1996, 2024)
]["sentence"].dropna().shape[0]

print("Total UNIQUE sentences (1996–2024):", total_sentences)

Total UNIQUE sentences (1996–2024): 47374


In [1]:
# =========================================================
# Google Colab (GPU) | ClimateBERT detector on UNIQUE sentences
# Model: climatebert/distilroberta-base-climate-detector
# Input : climate_finance_sentences_unique.csv  (in Drive folder)
# Output: climate_finance_sentences_unique_climatebert.csv
# Columns added:
#   bert_label      -> {"climate","non-climate"}
#   bert_score      -> P(predicted class)
#   bert_p_climate  -> P(climate) = P(yes)
# =========================================================

# !pip -q install -U transformers accelerate datasets tqdm

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive

# -------------------------
# 1) Mount Google Drive
# -------------------------
drive.mount('/content/drive')

# -------------------------
# 2) Paths (update folder if needed)
# -------------------------
FOLDER = "/content/drive/MyDrive/Climate Finance-CB"
IN_PATH  = f"{FOLDER}/climate_finance_sentences_unique.csv"
OUT_PATH = f"{FOLDER}/climate_finance_sentences_unique_climatebert.csv"

MODEL_NAME = "climatebert/distilroberta-base-climate-detector"
TEXT_COL   = "sentence"

# -------------------------
# 3) Device
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -------------------------
# 4) Load data
# -------------------------
df = pd.read_csv(IN_PATH)

if TEXT_COL not in df.columns:
    raise ValueError(f"Missing column '{TEXT_COL}'. Found: {list(df.columns)}")

texts = df[TEXT_COL].fillna("").astype(str).tolist()
print(f"Loaded rows: {len(df):,}")

# -------------------------
# 5) Load model/tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

id2label = model.config.id2label  # expected {0: "no", 1: "yes"}
print("Model id2label:", id2label)

# -------------------------
# 6) Batched inference
# -------------------------
BATCH_SIZE = 128
MAX_LEN    = 256

bert_label = []
bert_score = []
bert_p_climate = []

softmax = torch.nn.Softmax(dim=-1)

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Scoring"):
    batch_texts = texts[start:start + BATCH_SIZE]

    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = softmax(logits)           # [B, 2]
        p_yes = probs[:, 1].detach().cpu()  # "yes" => climate

        pred_id = torch.argmax(probs, dim=-1).detach().cpu()
        pred_prob = torch.gather(probs.detach().cpu(), 1, pred_id.unsqueeze(1)).squeeze(1)

    for pid, p_pred, p_cl in zip(pred_id.tolist(), pred_prob.tolist(), p_yes.tolist()):
        raw = str(id2label.get(pid, pid)).lower()
        if raw == "yes":
            bert_label.append("climate")
        else:
            bert_label.append("non-climate")

        bert_score.append(float(p_pred))
        bert_p_climate.append(float(p_cl))

# -------------------------
# 7) Attach outputs + print summary
# -------------------------
df["bert_label"] = bert_label
df["bert_score"] = bert_score
df["bert_p_climate"] = bert_p_climate

total = len(df)
n_climate = int((df["bert_label"] == "climate").sum())
n_non = total - n_climate

print("\n=== ClimateBERT results (sentence-level) ===")
print(f"Total sentences:        {total:,}")
print(f"Predicted climate:      {n_climate:,} ({n_climate/total:.2%})")
print(f"Predicted non-climate:  {n_non:,} ({n_non/total:.2%})")

# -------------------------
# 8) Save
# -------------------------
df.to_csv(OUT_PATH, index=False, encoding="utf-8")
print(f"\nSaved: {OUT_PATH}")
print(f"Rows saved: {len(df):,}")

Mounted at /content/drive
Using device: cuda
Loaded rows: 50,251


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/887 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: climatebert/distilroberta-base-climate-detector
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model id2label: {0: 'no', 1: 'yes'}


Scoring:   0%|          | 0/393 [00:00<?, ?it/s]


=== ClimateBERT results (sentence-level) ===
Total sentences:        50,251
Predicted climate:      30,430 (60.56%)
Predicted non-climate:  19,821 (39.44%)

Saved: /content/drive/MyDrive/Climate Finance-CB/climate_finance_sentences_unique_climatebert.csv
Rows saved: 50,251


In [2]:
# Climate-only filter (keeps bert_label == "climate" and overwrites the same file)

import pandas as pd

FOLDER = "/content/drive/MyDrive/Climate Finance-CB"
PATH = f"{FOLDER}/climate_finance_sentences_unique_climatebert.csv"

# Load
df = pd.read_csv(PATH)

# Keep only predicted climate
df_climate = df[df["bert_label"].astype(str).str.strip().str.lower() == "climate"].copy()

# Print summary
total = len(df)
kept = len(df_climate)
dropped = total - kept

print("=== Filter to Climate-only (overwrite) ===")
print(f"Total rows before: {total:,}")
print(f"Kept (climate):    {kept:,} ({kept/total:.2%})")
print(f"Dropped:           {dropped:,} ({dropped/total:.2%})")

# Overwrite the same file
df_climate.to_csv(PATH, index=False, encoding="utf-8")

print(f"\nOverwritten file saved at:\n{PATH}")
print(f"Rows now in file: {kept:,}")

=== Filter to Climate-only (overwrite) ===
Total rows before: 50,251
Kept (climate):    30,430 (60.56%)
Dropped:           19,821 (39.44%)

Overwritten file saved at:
/content/drive/MyDrive/Climate Finance-CB/climate_finance_sentences_unique_climatebert.csv
Rows now in file: 30,430


In [ ]:
import pandas as pd

# Read (notebook and data file in the same folder)
df = pd.read_csv("climate_finance_sentences_unique_climatebert_final.csv")

# Date parsing (two-pass) + parsed date column
d1 = pd.to_datetime(df["date"], errors="coerce")
d2 = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)
df["date_parsed"] = d1.fillna(d2)

# Guardrail: stop if any dates remain unparsed
if df["date_parsed"].isna().any():
    bad = (
        df.loc[df["date_parsed"].isna(), "date"]
          .dropna()
          .astype(str)
          .head(20)
          .to_list()
    )
    raise ValueError(f"Unparsed dates remain. Examples: {bad}")

# Filter to 1996–2024
df_1996_2024 = df[df["date_parsed"].dt.year.between(1996, 2024)].copy()

# Optional: keep only unique sentences (uncomment if needed)
# df_1996_2024 = (
#     df_1996_2024
#     .dropna(subset=["sentence"])
#     .drop_duplicates(subset=["sentence"])
# )

# Save CSV in the same folder as the notebook
out_path = "climate_speeches_data_final_1996_2024.csv"
df_1996_2024.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print("Rows saved:", len(df_1996_2024))
print("Non-missing sentences:", df_1996_2024["sentence"].notna().sum())

Saved: climate_speeches_data_final_1996_2024.csv
Rows saved: 24898
Non-missing sentences: 24898
